#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

In [ ]:
#@title 1. Initialize Core Environment
#@markdown This prepares the ephemeral storage, installs ComfyUI, and configures the MOSS-Image integration tools.

import os
import subprocess
from IPython.display import clear_output

LOCAL_WORKSPACE = "/content/ComfyUI"

print("🚀 [@CoinNoin] Initializing Core Architecture...")
if not os.path.exists(LOCAL_WORKSPACE):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE} &> /dev/null
    print("   ✓ Core Engine Cloned")
else:
    !cd {LOCAL_WORKSPACE} && git pull &> /dev/null
    print("   ✓ Core Engine Updated")

print("📦 [@CoinNoin] Installing Dependencies (This takes a moment)...")
!pip install -q uv
!cd {LOCAL_WORKSPACE} && uv pip install --system -q -r requirements.txt
!uv pip install --system -q xformers!=0.0.18 --extra-index-url https://download.pytorch.org/whl/cu121

clear_output()
print("✅ [@CoinNoin] Environment Ready!")

In [ ]:
#@title 2. High-Speed Asset Downloader & LoRA Setup
#@markdown The **MOSS-Image checkpoints** (SFT & Base) and required base assets are fetched automatically. You can optionally download or upload extra LoRAs.

# --- Model Selection ---
MODEL_DOWNLOAD = "SFT only" #@param ["Both (SFT + Base)", "SFT only", "Base only"]

# --- Optional LoRA Source ---
LORA_SOURCE = "None (Use Default Edit LoRA)" #@param ["None (Use Default Edit LoRA)", "Download from URL", "Upload from Computer"]
LORA_URLS = "" #@param {type:"string"}

import os
import subprocess
import urllib.parse
from google.colab import files

WORKSPACE = "/content/ComfyUI"

# Fixed URLs for MOSS-Image
SFT_UNET_URL = "https://huggingface.co/Comfy-Org/MOSS-Image/resolve/main/diffusion_models/moss_image_sft_int8_convrot.safetensors"
BASE_UNET_URL = "https://huggingface.co/Comfy-Org/MOSS-Image/resolve/main/diffusion_models/moss_image_base_int8_convrot.safetensors"
TEXT_ENCODER_URL = "https://huggingface.co/Comfy-Org/MOSS-Image/resolve/main/text_encoders/qwen3.5_9b_w4a8_gptq.safetensors"
VAE_URL = "https://huggingface.co/Comfy-Org/MOSS-Image/resolve/main/vae/flux2-vae_small.safetensors"
DEFAULT_EDIT_LORA_URL = "https://huggingface.co/Comfy-Org/MOSS-Image/resolve/main/loras/moss_image_edit_lora_rank_64_bf16.safetensors"

DIRS = {
    "unet":  os.path.join(WORKSPACE, "models/diffusion_models"),
    "clip":  os.path.join(WORKSPACE, "models/text_encoders"),
    "vae":   os.path.join(WORKSPACE, "models/vae"),
    "loras": os.path.join(WORKSPACE, "models/loras"),
    "input": os.path.join(WORKSPACE, "input"),
}

for p in DIRS.values():
    os.makedirs(p, exist_ok=True)

print("⚡ [@CoinNoin] Configuring Aria2c Accelerator...")
subprocess.run(['apt-get', '-y', 'install', '-qq', 'aria2'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def download_file(url, target_dir, filename=None):
    if not url.strip(): return
    if not filename:
        parsed_url = urllib.parse.urlparse(url)
        filename = os.path.basename(parsed_url.path)
    dest = os.path.join(target_dir, filename)
    if os.path.exists(dest) and os.path.getsize(dest) > 1000000:
        print(f"   ✅ Already Cached: {filename}")
        return

    print(f"   📥 Fetching: {filename}...")
    aria2_cmd = [
        "aria2c", "--console-log-level=error", "--summary-interval=10",
        "-c", "-x", "16", "-s", "16", "-k", "1M",
        "-o", filename, url, "-d", target_dir
    ]
    subprocess.run(aria2_cmd, check=True)
    print(f"   ✅ Saved in {os.path.basename(target_dir)}")

# Download selected UNET checkpoint(s)
if MODEL_DOWNLOAD in ("Both (SFT + Base)", "SFT only"):
    download_file(SFT_UNET_URL, DIRS["unet"])
if MODEL_DOWNLOAD in ("Both (SFT + Base)", "Base only"):
    download_file(BASE_UNET_URL, DIRS["unet"])

# Required shared assets
download_file(TEXT_ENCODER_URL, DIRS["clip"])
download_file(VAE_URL, DIRS["vae"])
download_file(DEFAULT_EDIT_LORA_URL, DIRS["loras"])

# Handle optional extra LoRA
ACTIVE_LORA_FILENAME = "moss_image_edit_lora_rank_64_bf16.safetensors"

if LORA_SOURCE == "Download from URL":
    if LORA_URLS.strip():
        parsed_name = os.path.basename(urllib.parse.urlparse(LORA_URLS.strip()).path)
        if not parsed_name.endswith(('.safetensors', '.pt', '.ckpt')):
            parsed_name = "downloaded_lora.safetensors"
        download_file(LORA_URLS.strip(), DIRS["loras"], filename=parsed_name)
        ACTIVE_LORA_FILENAME = parsed_name
    else:
        print("⚠️ No LoRA URL provided. Using default.")
elif LORA_SOURCE == "Upload from Computer":
    print("\n📤 [@CoinNoin] Select your extra LoRA (.safetensors) file to upload:")
    uploaded = files.upload()
    if uploaded:
        for fname in uploaded.keys():
            dest = os.path.join(DIRS["loras"], fname)
            os.replace(fname, dest)
            ACTIVE_LORA_FILENAME = fname
            print(f"   ✅ Extra LoRA uploaded: {fname}")
    else:
        print("⚠️ No file uploaded. Using default.")

available_loras = [f for f in os.listdir(DIRS["loras"]) if f.endswith(('.safetensors', '.pt', '.ckpt'))]
print("\n📂 [@CoinNoin] Available LoRAs in storage:")
for idx, lora_name in enumerate(available_loras, 1):
    print(f"   {idx}. {lora_name}")

print("\n✅ [@CoinNoin] All core assets secured!")

In [ ]:
#@title 3. MOSS-Image Text-to-Image Generation
#@markdown Describe the finished image in your prompt and adjust generation settings.

# --- Model Selection ---
MODEL_CHOICE = "SFT (instruction-tuned)" #@param ["SFT (instruction-tuned)", "Base (raw)"]

# --- Generation Settings ---
PROMPT = "This is a candid photograph taken with a smartphone of a person sitting on a concrete ledge beneath a white slatted roof, gazing toward the left at a sprawling cityscape. The individual wears a black graphic t-shirt under an unbuttoned dark patterned short-sleeve shirt and tan cargo pants, with a silver watch on their wrist and a large blue tote bag resting beside them. Behind them, a metal railing separates the foreground from a dense urban grid of skyscrapers and low-rise buildings stretching toward a wide body of water dotted with boats and an island in the distance under a bright blue sky scattered with fluffy white clouds." #@param {type:"string"}
NEGATIVE_PROMPT = "nsfw, worst quality, bad anatomy, watermark, jpeg artifact" #@param {type:"string"}
WIDTH = 512 #@param {type:"slider", min:512, max:2048, step:16}
HEIGHT = 512 #@param {type:"slider", min:512, max:2048, step:16}

# --- LoRA Selection ---
#@markdown *Guide: Choose **None** for no LoRA, **auto** to use the most recent LoRA from Cell 2, or type the exact filename.*
LORA_FILENAME = "None" #@param ["None", "auto"] {allow-input: true}
LORA_STRENGTH = 1 #@param {type:"slider", min:0.0, max:2.0, step:0.05}

# --- Advanced Sampler Settings ---
STEPS = 50 #@param {type:"slider", min:10, max:50, step:1}
CFG = 3 #@param {type:"number"}
SAMPLER_NAME = "euler" #@param ["er_sde", "euler", "euler_ancestral", "dpmpp_2m", "dpmpp_sde", "dpmpp_2m_sde", "uni_pc", "res_multistep", "lcm"] {allow-input: true}
SCHEDULER = "simple" #@param ["simple", "normal", "karras", "exponential", "sgm_uniform", "ddim_uniform", "beta"] {allow-input: true}
SEED = 0 #@param {type:"integer"}

# --- Guidance Options ---
ENABLE_SLG = True #@param {type:"boolean"}
#@markdown *Skip Layer Guidance (layer 22). Turning it off looks more realistic but can collapse detail.*

# --- Direct Download (No ZIP) ---
AUTO_DOWNLOAD = False #@param {type:"boolean"}

import sys
import os
import json
import time
import random
import subprocess
import urllib.request
from google.colab import files
from PIL import Image
from IPython.display import display, Image as IPImage, clear_output

WORKSPACE = "/content/ComfyUI"
os.chdir(WORKSPACE)

# Model mapping
MODEL_MAP = {
    "SFT (instruction-tuned)": "moss_image_sft_int8_convrot.safetensors",
    "Base (raw)": "moss_image_base_int8_convrot.safetensors",
}
UNET_FILENAME = MODEL_MAP[MODEL_CHOICE]

# Auto-detect LoRA
if LORA_FILENAME.lower() == "auto":
    if 'ACTIVE_LORA_FILENAME' in globals() and ACTIVE_LORA_FILENAME:
        LORA_FILENAME = ACTIVE_LORA_FILENAME
    else:
        LORA_FILENAME = "moss_image_edit_lora_rank_64_bf16.safetensors"

if SEED == 0:
    SEED = random.randint(1, 1125899906842624)

print("🔌 [@CoinNoin] Checking ComfyUI Server Status...")
def start_server():
    req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
    try:
        urllib.request.urlopen(req)
        print("   🟢 Server is already running.")
    except:
        print("   🚀 Starting ComfyUI Server in background...")
        subprocess.Popen([sys.executable, "main.py", "--port", "8188", "--lowvram"],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        while True:
            try:
                urllib.request.urlopen(req)
                print("   🟢 Server is now up and ready!")
                break
            except:
                time.sleep(2)

start_server()
clear_output()

BRAND_PREFIX = "@CoinNoin_MOSST2I"

print(f"\033[94m➜ [@CoinNoin] Details | Model: {MODEL_CHOICE} | Res: {WIDTH}x{HEIGHT} | Seed: {SEED} | Steps: {STEPS} | CFG: {CFG}\033[0m")

# Build workflow
prompt_workflow = {
    "unet": {"class_type": "UNETLoader",
             "inputs": {"unet_name": UNET_FILENAME, "weight_dtype": "default"}},
    "clip": {"class_type": "CLIPLoader",
             "inputs": {"clip_name": "qwen3.5_9b_w4a8_gptq.safetensors", "type": "moss_image", "device": "default"}},
    "vae":  {"class_type": "VAELoader", "inputs": {"vae_name": "flux2-vae_small.safetensors"}},
    "pos":  {"class_type": "CLIPTextEncode",
             "inputs": {"clip": ["clip", 0], "text": PROMPT}},
    "neg":  {"class_type": "CLIPTextEncode",
             "inputs": {"clip": ["clip", 0], "text": NEGATIVE_PROMPT}},
    "latent": {"class_type": "EmptyFlux2LatentImage",
               "inputs": {"width": WIDTH, "height": HEIGHT, "batch_size": 1}},
}

model_output = ["unet", 0]

# Optional user LoRA
if LORA_FILENAME.lower() != "none" and LORA_STRENGTH > 0:
    prompt_workflow["lora"] = {
        "class_type": "LoraLoaderModelOnly",
        "inputs": {"lora_name": LORA_FILENAME,
                   "strength_model": LORA_STRENGTH,
                   "model": ["unet", 0]}
    }
    model_output = ["lora", 0]
    print(f"✨ [@CoinNoin] LoRA Applied: {LORA_FILENAME} (Strength: {LORA_STRENGTH})")

# Optional Skip Layer Guidance
if ENABLE_SLG:
    prompt_workflow["slg"] = {
        "class_type": "SkipLayerGuidanceDiTSimple",
        "inputs": {
            "double_layers": "",
            "single_layers": "22",
            "start_percent": 0.0,
            "end_percent": 0.5,
            "model": model_output
        }
    }
    model_output = ["slg", 0]
    print("✨ [@CoinNoin] Skip Layer Guidance enabled (layer 22)")

# KSampler
prompt_workflow["sampler"] = {
    "class_type": "KSampler",
    "inputs": {
        "seed": SEED, "steps": STEPS, "cfg": CFG,
        "sampler_name": SAMPLER_NAME, "scheduler": SCHEDULER, "denoise": 1.0,
        "model": model_output,
        "positive": ["pos", 0], "negative": ["neg", 0],
        "latent_image": ["latent", 0]
    }
}

prompt_workflow["decode"] = {"class_type": "VAEDecode",
                             "inputs": {"samples": ["sampler", 0], "vae": ["vae", 0]}}
prompt_workflow["save"] = {"class_type": "SaveImage",
                           "inputs": {"filename_prefix": BRAND_PREFIX, "images": ["decode", 0]}}

# Submit
p = {"prompt": prompt_workflow}
data = json.dumps(p).encode('utf-8')
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)

print("\n📥 Submitting workflow to API...")
try:
    response = urllib.request.urlopen(req)
    prompt_id = json.loads(response.read())['prompt_id']
except Exception as e:
    print(f"❌ API Error: {e}")
    raise

print("✨ Processing and Sampling (Check ComfyUI server logs if stuck)...")
while True:
    try:
        history_req = urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}")
        history_res = urllib.request.urlopen(history_req)
        history_data = json.loads(history_res.read())
        if prompt_id in history_data:
            outputs = history_data[prompt_id]['outputs']
            break
    except:
        pass
    time.sleep(1)

print("🖼️ Decoding Final Masterpiece...")
generated_images = []
for node_id, node_output in outputs.items():
    if 'images' in node_output:
        for image in node_output['images']:
            filename = image['filename']
            img_path = os.path.join(WORKSPACE, "output", filename)
            generated_images.append(img_path)
            print(f"\033[92m✓ [@CoinNoin] Saved: {filename}\033[0m")
            display(IPImage(filename=img_path))

if AUTO_DOWNLOAD and generated_images:
    print("\n📥 [@CoinNoin] Download Started...")
    for img_file in generated_images:
        files.download(img_file)

In [ ]:
#@title 4. MOSS-Image Edit
#@markdown Upload a source image and describe only the change. The Edit LoRA redraws the same frame with the described modification applied.

# --- Upload Controls ---
UPLOAD_SOURCE_IMAGE = True #@param {type:"boolean"}
#@markdown *If checked, Colab will prompt you to upload an image when you run this cell. Uncheck to use previously uploaded.*

# --- Model Selection ---
MODEL_CHOICE = "SFT (instruction-tuned)" #@param ["SFT (instruction-tuned)", "Base (raw)"]

# --- Edit Settings ---
EDIT_INSTRUCTION = "Change only the shirt to a black leather jacket. Keep the character, face, hair, pose, background and composition unchanged." #@param {type:"string"}
NEGATIVE_PROMPT = "nsfw, worst quality, bad anatomy, watermark, jpeg artifact" #@param {type:"string"}

# --- LoRA Selection ---
#@markdown *The **core Edit LoRA is always applied**. This extra LoRA sits on top of it. Choose **None** or **auto**, or type the exact filename.*
LORA_FILENAME = "None" #@param ["None", "auto"] {allow-input: true}
LORA_STRENGTH = 1.0 #@param {type:"slider", min:0.0, max:2.0, step:0.05}

# --- Advanced Sampler Settings ---
STEPS = 30 #@param {type:"slider", min:10, max:50, step:1}
CFG = 4.0 #@param {type:"number"}
#@markdown *Keep CFG at 4.0 – the negative branch matters for this LoRA.*
SEED = 0 #@param {type:"integer"}

# --- Direct Download (No ZIP) ---
AUTO_DOWNLOAD = False #@param {type:"boolean"}

import sys
import os
import json
import time
import random
import subprocess
import urllib.request
from google.colab import files
from PIL import Image
from IPython.display import display, Image as IPImage, clear_output

WORKSPACE = "/content/ComfyUI"
INPUT_DIR = os.path.join(WORKSPACE, "input")
os.makedirs(INPUT_DIR, exist_ok=True)
os.chdir(WORKSPACE)

MODEL_MAP = {
    "SFT (instruction-tuned)": "moss_image_sft_int8_convrot.safetensors",
    "Base (raw)": "moss_image_base_int8_convrot.safetensors",
}
UNET_FILENAME = MODEL_MAP[MODEL_CHOICE]

if LORA_FILENAME.lower() == "auto":
    if 'ACTIVE_LORA_FILENAME' in globals() and ACTIVE_LORA_FILENAME:
        LORA_FILENAME = ACTIVE_LORA_FILENAME
    else:
        LORA_FILENAME = "moss_image_edit_lora_rank_64_bf16.safetensors"

if SEED == 0:
    SEED = random.randint(1, 1125899906842624)

print("🔌 [@CoinNoin] Checking ComfyUI Server Status...")
def start_server():
    req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
    try:
        urllib.request.urlopen(req)
        print("   🟢 Server is already running.")
    except:
        print("   🚀 Starting ComfyUI Server in background...")
        subprocess.Popen([sys.executable, "main.py", "--port", "8188", "--lowvram"],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        while True:
            try:
                urllib.request.urlopen(req)
                print("   🟢 Server is now up and ready!")
                break
            except:
                time.sleep(2)

start_server()
clear_output()

# Handle upload
source_filename = "moss_edit_source.png"
if UPLOAD_SOURCE_IMAGE:
    print("📤 [@CoinNoin] Please upload your SOURCE image for editing...")
    uploaded = files.upload()
    if uploaded:
        source_name = list(uploaded.keys())[0]
        os.replace(source_name, os.path.join(INPUT_DIR, source_filename))
        print("   ✅ Source image saved!")
        saved_path = os.path.join(INPUT_DIR, source_filename)
        img = Image.open(saved_path)
        display(img.resize((300, int(300 * img.height / img.width))))
    else:
        print("   ❌ No image uploaded. Will attempt to use cached image.")
else:
    print("   ℹ️ Using previously uploaded source image.")

BRAND_PREFIX = "@CoinNoin_MOSSEdit"

print(f"\n\033[94m➜ [@CoinNoin] Details | Model: {MODEL_CHOICE} | Seed: {SEED} | Steps: {STEPS} | CFG: {CFG}\033[0m")
print(f"📝 Instruction: '{EDIT_INSTRUCTION}'")

# Build workflow
prompt_workflow = {
    "unet": {"class_type": "UNETLoader",
             "inputs": {"unet_name": UNET_FILENAME, "weight_dtype": "default"}},
    "clip": {"class_type": "CLIPLoader",
             "inputs": {"clip_name": "qwen3.5_9b_w4a8_gptq.safetensors", "type": "moss_image", "device": "default"}},
    "vae":  {"class_type": "VAELoader", "inputs": {"vae_name": "flux2-vae_small.safetensors"}},
}

# Core Edit LoRA (always applied)
prompt_workflow["lora_edit"] = {
    "class_type": "LoraLoaderModelOnly",
    "inputs": {
        "lora_name": "moss_image_edit_lora_rank_64_bf16.safetensors",
        "strength_model": 1.0,
        "model": ["unet", 0]
    }
}
model_output = ["lora_edit", 0]

# Optional extra user LoRA (applied on top of edit LoRA)
if LORA_FILENAME.lower() != "none" and LORA_STRENGTH > 0:
    if LORA_FILENAME != "moss_image_edit_lora_rank_64_bf16.safetensors":
        prompt_workflow["lora_user"] = {
            "class_type": "LoraLoaderModelOnly",
            "inputs": {
                "lora_name": LORA_FILENAME,
                "strength_model": LORA_STRENGTH,
                "model": ["lora_edit", 0]
            }
        }
        model_output = ["lora_user", 0]
        print(f"✨ [@CoinNoin] Extra LoRA Applied: {LORA_FILENAME} (Strength: {LORA_STRENGTH})")

# Image loading & scaling
prompt_workflow["load"] = {"class_type": "LoadImage", "inputs": {"image": source_filename}}
prompt_workflow["scale"] = {
    "class_type": "ImageScaleToTotalPixels",
    "inputs": {
        "image": ["load", 0],
        "upscale_method": "lanczos",
        "megapixels": 1.0,
        "resolution_steps": 16
    }
}
prompt_workflow["size"] = {"class_type": "GetImageSize", "inputs": {"image": ["scale", 0]}}
prompt_workflow["latent"] = {
    "class_type": "EmptyFlux2LatentImage",
    "inputs": {"width": ["size", 0], "height": ["size", 1], "batch_size": 1}
}

# MOSS-Image Edit encoders
prompt_workflow["pos"] = {
    "class_type": "TextEncodeMOSSImageEdit",
    "inputs": {
        "clip": ["clip", 0],
        "vae": ["vae", 0],
        "images.image_0": ["scale", 0],
        "prompt": EDIT_INSTRUCTION,
        "edit_format": "Moss-Image-Edit",
        "image_strength": 1.0
    }
}
prompt_workflow["neg"] = {
    "class_type": "TextEncodeMOSSImageEdit",
    "inputs": {
        "clip": ["clip", 0],
        "vae": ["vae", 0],
        "images.image_0": ["scale", 0],
        "prompt": NEGATIVE_PROMPT,
        "edit_format": "Moss-Image-Edit",
        "image_strength": 1.0
    }
}

prompt_workflow["sampler"] = {
    "class_type": "KSampler",
    "inputs": {
        "seed": SEED, "steps": STEPS, "cfg": CFG,
        "sampler_name": "euler", "scheduler": "normal", "denoise": 1.0,
        "model": model_output,
        "positive": ["pos", 0], "negative": ["neg", 0],
        "latent_image": ["latent", 0]
    }
}

prompt_workflow["decode"] = {"class_type": "VAEDecode",
                             "inputs": {"samples": ["sampler", 0], "vae": ["vae", 0]}}
prompt_workflow["save"] = {"class_type": "SaveImage",
                           "inputs": {"filename_prefix": BRAND_PREFIX, "images": ["decode", 0]}}

# Submit
p = {"prompt": prompt_workflow}
data = json.dumps(p).encode('utf-8')
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)

print("\n📥 Submitting workflow to API...")
try:
    response = urllib.request.urlopen(req)
    prompt_id = json.loads(response.read())['prompt_id']
except Exception as e:
    print(f"❌ API Error: {e}")
    raise

print("✨ Processing and Sampling (Check ComfyUI server logs if stuck)...")
while True:
    try:
        history_req = urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}")
        history_res = urllib.request.urlopen(history_req)
        history_data = json.loads(history_res.read())
        if prompt_id in history_data:
            outputs = history_data[prompt_id]['outputs']
            break
    except:
        pass
    time.sleep(1)

print("🖼️ Decoding Final Masterpiece...")
generated_images = []
for node_id, node_output in outputs.items():
    if 'images' in node_output:
        for image in node_output['images']:
            filename = image['filename']
            img_path = os.path.join(WORKSPACE, "output", filename)
            generated_images.append(img_path)
            print(f"\033[92m✓ [@CoinNoin] Saved: {filename}\033[0m")
            display(IPImage(filename=img_path))

if AUTO_DOWNLOAD and generated_images:
    print("\n📥 [@CoinNoin] Download Started...")
    for img_file in generated_images:
        files.download(img_file)